In [ ]:
#Sratch work to get inlcude directory and .h files generated or copied as needed

In [9]:
# imports 
# set up paths and server

import datetime
import sys
import os
import pyflow as pf
scratchdir = os.path.join(os.path.abspath(''), 'scratch_include')
filesdir = os.path.join(scratchdir, 'files')
outdir = os.path.join(scratchdir, 'out')

if not os.path.exists(outdir):
    os.makedirs(outdir, exist_ok=True)
    
server_host = 'localhost'
server_port = 22921 #Anna's personal EcFlow server port

In [ ]:
# Create basic suite

class WorkflowTask(pf.Task):
    def __init__(self, name: str, context: dict, **kwargs):
        super().__init__(name, variables=context['variables'], script=context['script'])
    
class TestSuiteBuilder:
    def __init__(self, filesdir, outdir, number=100):
        with pf.Suite('testSuite', host=pf.LocalHost('localhost'),
                      files=os.path.join(filesdir, 'testSuite', 'scripts'),
                      home=outdir, NUMBER=number) as s: #remove NUMBER?

            # Create family_Aa and its tasks
            with pf.AnchorFamily('A'):
                tAa1c = {'variables': {'NUMBER': number + 1}, #c is context
                            'script': "echo family_A NUMBER=$NUMBER"}
                tAa1 = WorkflowTask('A', tAa1c) #this will be py ecflow. context is a dict

        self.s = s
        self.s.check_definition()
        self.s.deploy_suite()
        
        # Create def directory and save suite definition there
        def_dir = os.path.join(filesdir, 'testSuite', 'def')
        if not os.path.exists(def_dir):
            os.makedirs(def_dir, exist_ok=True)
        suite_def = self.s.ecflow_definition()
        # Save definition to def directory
        suite_def.save_as_defs(os.path.join(def_dir, 'testSuite.def'))
        # Create include directory in the testSuite directory
        include_dir = os.path.join(filesdir, 'testSuite', 'include')
        if not os.path.exists(include_dir):
            os.makedirs(include_dir, exist_ok=True)
        
        

# Instantiate using default 100
builder = TestSuiteBuilder(filesdir, outdir, 100)
print(builder.s)

suite testSuite
  edit NUMBER '100'
  edit ECF_FILES '/scratch4/NCEPDEV/global/Anna.Smoot/Pyflow/pyflow/tutorials/course/scratch/scratch_include/files/testSuite/scripts'
  edit ECF_HOME '/scratch4/NCEPDEV/global/Anna.Smoot/Pyflow/pyflow/tutorials/course/scratch/scratch_include/out'
  edit ECF_JOB_CMD 'bash -c 'export ECF_PORT=%ECF_PORT%; export ECF_HOST=%ECF_HOST%; export ECF_NAME=%ECF_NAME%; export ECF_PASS=%ECF_PASS%; export ECF_TRYNO=%ECF_TRYNO%; export PATH=/home/Anna.Smoot/.conda/envs/pf3.5.1/bin:$PATH; ecflow_client --init="$$" && %ECF_JOB% && ecflow_client --complete || ecflow_client --abort ' 1> %ECF_JOBOUT% 2>&1 &'
  edit ECF_KILL_CMD 'pkill -15 -P %ECF_RID%'
  edit ECF_STATUS_CMD 'true'
  edit ECF_CHECK_CMD 'true'
  edit ECF_OUT '%ECF_HOME%'
  label exec_host "localhost"
  family A
    edit ECF_FILES '/scratch4/NCEPDEV/global/Anna.Smoot/Pyflow/pyflow/tutorials/course/scratch/scratch_include/files/testSuite/scripts/A'
    task A
      edit NUMBER '101'
  endfamily
endsuite

